In [6]:
from dotenv import load_dotenv
load_dotenv("../.env")

import os
from pathlib import Path

# Gemini embedding model — same one we used during indexing
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Gemini LLM — this is what generates the final answer
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain's Chroma wrapper — to load our saved vector store
from langchain_chroma import Chroma

# PromptTemplate lets us write a reusable prompt with placeholders
from langchain_core.prompts import PromptTemplate

# RunnablePassthrough passes the user's question through the chain unchanged
from langchain_core.runnables import RunnablePassthrough

# StrOutputParser converts Gemini's response object into a plain string
from langchain_core.output_parsers import StrOutputParser

api_key = os.getenv("GOOGLE_API_KEY")
CHROMA_DIR = Path("../chroma_db")

print(f"API key loaded: {api_key[:8]}...")
print(f"ChromaDB path exists: {CHROMA_DIR.exists()}")


API key loaded: AQ.Ab8RN...
ChromaDB path exists: True


In [7]:
# Set up the same embedding model we used during indexing
# IMPORTANT: must be exactly the same model — otherwise the vectors won't match
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=api_key
)

# Load the existing ChromaDB vector store from disk
# This does NOT re-embed anything — it just opens the saved database
vectorstore = Chroma(
    collection_name="ml_papers",       # same collection name we used in Step 3
    persist_directory=str(CHROMA_DIR), # same folder we saved to in Step 3
    embedding_function=embeddings      # needed so Chroma knows how to embed new queries
)

# Create a retriever from the vector store
# k=5 means: return the 5 most relevant chunks for any given question
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

# Quick test — let's see what chunks come back for a sample question
test_docs = retriever.invoke("What is a residual connection?")
print(f"Retrieved {len(test_docs)} chunks\n")

# Print the source and page of each retrieved chunk
for doc in test_docs:
    print(f"Source: {doc.metadata['source']} | Page: {doc.metadata['page']}")



Retrieved 5 chunks

Source: resnet | Page: 3
Source: resnet | Page: 2
Source: resnet | Page: 2
Source: resnet | Page: 3
Source: resnet | Page: 2


In [12]:
# This is the heart of our prompt engineering
# {context} will be replaced with the retrieved chunks
# {question} will be replaced with the user's actual question

prompt_template = """
You are a helpful research assistant that explains machine learning papers 
to third-year undergraduate students in plain English.

Use ONLY the information provided in the context below to answer the question.
Do NOT use any outside knowledge.

For every piece of information you use, cite the source paper and page number 
in this format: [Paper: source_name, Page: X]

If the context contains mathematical notation or equations that are hard to read,
describe what the equation means in plain English instead of copying it directly.

If the context does not contain enough information to answer the question,
say: "I don't have enough information in the indexed papers to answer this question."

Context:
{context}

Question:
{question}

Answer (in plain English, with citations):
"""

# Create a LangChain PromptTemplate object from the string above
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]  # the two placeholders we defined
)

print("Prompt template ready.")
print(f"Input variables: {prompt.input_variables}")


Prompt template ready.
Input variables: ['context', 'question']


In [13]:
# Set up Gemini 2.5 Flash as the answering LLM
# temperature=0.2 means answers are mostly factual with a little flexibility
# (0 = fully deterministic, 1 = very creative — for research Q&A we want low temperature)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.2
)

# This helper function takes the list of retrieved Document objects
# and formats them into a single string that fits into the {context} placeholder
def format_docs(docs):
    formatted=[]
    for doc in docs:
        # Each chunk shows its source paper, page number, and text content
        chunk_text = f"[Paper: {doc.metadata['source']}, Page: {doc.metadata['page']}]\n{doc.page_content}"
        formatted.append(chunk_text)
    # Join all chunks with a separator so Gemini can clearly see where one ends and another begins
    return "\n\n---\n\n".join(formatted)

# Build the RAG chain using LangChain's pipe operator (|)
# Read this left to right — it shows exactly how data flows:
# 1. Take the question → retrieve relevant docs AND pass question through unchanged
# 2. Format the docs into a string, keep the question
# 3. Slot both into the prompt template
# 4. Send the filled prompt to Gemini
# 5. Parse Gemini's response into a plain string
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")


RAG chain built successfully.


In [14]:
# test
question = "What problem does the Transformer architecture solve that previous models couldn't?"

print(f"Question: {question}")
print("="*60)

# Invoke the full RAG chain — this triggers the entire pipeline
answer = rag_chain.invoke(question)

print(answer)

Question: What problem does the Transformer architecture solve that previous models couldn't?
The Transformer architecture solves several problems present in previous models:

1.  **Limited Parallelization and Slow Training:** Previous models, particularly those relying on recurrence, had a "sequential nature" that prevented parallelization during training, especially with longer sequences due to memory limitations [Paper: attention_is_all_you_need, Page: 2]. The Transformer addresses this by "eschewing recurrence" and "dispensing with recurrence and convolutions entirely," instead relying "entirely on an attention mechanism" [Paper: attention_is_all_you_need, Page: 1, 2]. This design allows for "significantly more parallelization" and "significantly less time to train" [Paper: attention_is_all_you_need, Page: 1, 2].

2.  **Difficulty Learning Long-Range Dependencies:** Models like ConvS2S and ByteNet, which use convolutional neural networks, required a number of operations to relate s